# 01 Data Cleaning


In [ ]:
from pathlib import Path
import sys

def find_project_root(start: Path) -> Path:
    markers = ("src", "data", "notebooks")

    for p in [start, *start.parents]:
        if p.name == "customer-segmentation-analytics" and all((p / m).is_dir() for m in markers):
            return p

    for p in [start, *start.parents]:
        candidate = p / "Improvements" / "Statistics" / "customer-segmentation-analytics"
        if all((candidate / m).is_dir() for m in markers):
            return candidate

    raise RuntimeError("Could not locate customer-segmentation-analytics project root.")

PROJECT_ROOT = find_project_root(Path.cwd().resolve())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"PROJECT_ROOT: {PROJECT_ROOT}")


## Objective


Load customer-level data, validate schema quality, apply cleaning rules, and save a clean dataset.


In [ ]:
import pandas as pd

from src.data_cleaning import clean_customer_data, load_customer_data, quality_report

raw_path = PROJECT_ROOT / "data" / "raw" / "retail_customer_segmentation.csv"
df_raw = load_customer_data(raw_path)
print("Raw shape:", df_raw.shape)
df_raw.head()


## Data Quality Overview


In [ ]:
report = quality_report(df_raw)
report.sort_values("missing_pct", ascending=False).head(20)


## Clean Dataset


In [ ]:
df_clean = clean_customer_data(df_raw)
print("Clean shape:", df_clean.shape)
df_clean.head()


## Validation Checks


In [ ]:
assert df_clean["customer_id"].is_unique, "customer_id should be unique after cleaning"
assert (df_clean["discount_usage_rate"].between(0, 1)).all()
assert (df_clean["return_rate"].between(0, 1)).all()
assert (df_clean["months_active"] >= 0).all()
assert (df_clean["avg_monthly_spend"] >= 0).all()

checks = {
    "n_customers": len(df_clean),
    "n_segments": df_clean["customer_segment"].nunique(),
    "n_regions": df_clean["region"].nunique(),
}
checks


## Save Clean Dataset


In [ ]:
out_path = PROJECT_ROOT / "data" / "processed" / "customers_clean.parquet"
out_path.parent.mkdir(parents=True, exist_ok=True)
df_clean.to_parquet(out_path, index=False)
out_path
